In [1]:
import pandas as pd
import numpy as np
from pykalman import KalmanFilter

In [2]:
# df = pd.read_pickle("event_df_tidy_train")
# selected_columns = ["label", "signal", "duration"]
# df = df[selected_columns]
# df = df[df['label'] == 'flood']
# df.loc[:,"duration"] = df.apply(lambda x: math.ceil(x["signal"]["time"][-1]/60),axis=1)
# df = df.loc[df.duration > 4]

In [3]:
df = pd.read_pickle("flood_simulations")
df.head()

,deployment_id,label,signal,inflection_t,signal_sim,signal_sim_rise,signal_sim_fall
uuid,,,,,,,
1690089,daily_happy_satyr,flood,"{'time': [0, 567, 1009, 1639, 1765, 2080, 2395...",6961,"{'time': [0, 567, 1009, 1639, 1765, 2080, 2395...","{'time': [0, 567, 1009, 1639, 1765, 2080, 2395...","{'time': [6961, 7086, 7275, 7338, 7842, 7969, ..."
2578925,daily_happy_satyr,flood,"{'time': [0, 1889, 1952, 2960, 4914, 4921, 674...",7062,"{'time': [0, 1889, 1952, 2960, 4914, 4921, 674...","{'time': [0, 1889, 1952, 2960, 4914, 4921, 674...","{'time': [7062, 7692, 9014, 9580, 10588, 11091..."
5301386,daily_happy_satyr,flood,"{'time': [0, 1323, 1898, 1906, 2409, 2535, 278...",5303,"{'time': [0, 1323, 1898, 1906, 2409, 2535, 278...","{'time': [0, 1323, 1898, 1906, 2409, 2535, 278...","{'time': [5303, 5618, 6123, 6311, 6375, 6688, ..."
2808962,daily_happy_satyr,flood,"{'time': [126, 441, 503, 566, 882, 1008, 1071,...",4789,"{'time': [126, 441, 503, 566, 882, 1008, 1071,...","{'time': [126, 441, 503, 566, 882, 1008, 1071,...","{'time': [4789, 5292, 5796, 5986, 6174, 6237, ..."
9041605,daily_happy_satyr,flood,"{'time': [0, 379, 1449, 1575, 1890, 2394, 2709...",6188,"{'time': [0, 379, 1449, 1575, 1890, 2394, 2709...","{'time': [0, 379, 1449, 1575, 1890, 2394, 2709...","{'time': [6188, 6251, 6440, 6819, 7386, 7512, ..."


In [4]:
df['fall_length'] = df['signal_sim_fall'].apply(lambda x: len(x['time']) if 'time' in x else 0)
print(df['fall_length'].describe())

count     879.000000
mean      101.706485
std       164.523002
min         2.000000
25%        21.000000
50%        44.000000
75%       122.500000
max      1628.000000
Name: fall_length, dtype: float64


In [5]:
def preprocess_signal(signal):
    time = signal['time']
    depth = signal['depth']
    return pd.DataFrame({'time': time, 'depth': depth}).sort_values(by='time')

df['processed_signal'] = df['signal_sim_fall'].apply(preprocess_signal)
print(df['processed_signal'].iloc[0])

     time       depth
0    6961  265.394546
1    7086  265.255083
2    7275  265.036528
3    7338  264.960987
4    7842  264.278317
5    7969  264.074344
6    8788  262.133603
7    8851  261.915832
8    9103  260.894506
9    9733  256.821630
10  10236  251.032903
11  10615  244.298661
12  10930  236.541605
13  11434  218.701932
14  11938  192.422097
15  12568  146.455570
16  12693  135.844678
17  12882  119.198726
18  13197   90.832632
19  13260   85.224501
20  13512   63.663044


In [6]:
# Step 2: Vectorized Kalman Filtering with Variable Time Gaps
def vectorized_kalman_filter(signal):
    observations = signal['depth'].values
    times = signal['time'].values
    time_diffs = np.diff(times, prepend=times[0])  # Compute time gaps with 0 for the first element

    # Initialize state and covariance matrices
    n_observations = len(observations)
    state_dim = 2  # [depth, velocity]
    filtered_means = np.zeros((n_observations, state_dim))
    smoothed_means = np.zeros((n_observations, state_dim))
    covariances = np.zeros((n_observations, state_dim, state_dim))

    # Initial state and covariance
    state_mean = np.array([observations[0], 0])  # Initial [depth, velocity]
    state_covariance = np.eye(state_dim) * 1  # Initial uncertainty
    observation_matrix = np.array([[1, 0]])  # Observation model
    observation_covariance = np.array([[1]])  # Observation noise
    process_noise_base = np.array([[0.1, 0], [0, 0.1]])

    # Filtering
    for t in range(n_observations):
        if t > 0:
            delta_t = time_diffs[t - 1]
            transition_matrix = np.array([[1, delta_t], [0, 1]])
            process_noise = process_noise_base * delta_t

            # Predict step
            predicted_state_mean = np.dot(transition_matrix, state_mean)
            predicted_state_cov = (
                np.dot(transition_matrix, np.dot(state_covariance, transition_matrix.T)) + process_noise
            )

            # Update step
            innovation = observations[t] - np.dot(observation_matrix, predicted_state_mean)
            innovation_cov = (
                np.dot(observation_matrix, np.dot(predicted_state_cov, observation_matrix.T))
                + observation_covariance
            )
            kalman_gain = np.dot(
                predicted_state_cov,
                np.dot(observation_matrix.T, np.linalg.inv(innovation_cov))
            )
            state_mean = predicted_state_mean + np.dot(kalman_gain, innovation)
            state_covariance = predicted_state_cov - np.dot(
                kalman_gain, np.dot(observation_matrix, predicted_state_cov)
            )

        # Store filtered results
        filtered_means[t] = state_mean
        covariances[t] = state_covariance

    # Smoothing
    smoothed_means[-1] = filtered_means[-1]
    smoothed_covariance = covariances[-1]
    for t in range(n_observations - 2, -1, -1):
        delta_t = time_diffs[t]
        transition_matrix = np.array([[1, delta_t], [0, 1]])

        # RTS smoother gain
        predicted_covariance = (
            np.dot(transition_matrix, np.dot(covariances[t], transition_matrix.T)) + process_noise_base * delta_t
        )
        smoother_gain = np.dot(
            covariances[t],
            np.dot(transition_matrix.T, np.linalg.inv(predicted_covariance))
        )

        # Update smoothed state
        smoothed_means[t] = (
            filtered_means[t]
            + np.dot(smoother_gain, (smoothed_means[t + 1] - np.dot(transition_matrix, filtered_means[t])))
        )

    # Add filtered and smoothed results to the signal DataFrame
    signal['kalman_depth'] = filtered_means[:, 0]
    signal['smoothed_depth'] = smoothed_means[:, 0]
    return signal


In [7]:
df['filtered_signal'] = df['processed_signal'].apply(vectorized_kalman_filter)

In [8]:
print(df['filtered_signal'].iloc[8])

    time      depth  kalman_depth  smoothed_depth
0   1515  46.918837     46.918837       46.896059
1   1578  46.873299     46.896068       46.896059
2   1641  46.829576     46.829593       46.829595
3   1704  46.787528     46.787527       46.787527
4   1767  46.747029     46.747029       46.747029
..   ...        ...           ...             ...
57  5122  20.819903     20.819907       20.819903
58  5185  18.335798     18.335800       18.335798
59  5248  15.850590     15.850590       15.850590
60  5311  13.416809     13.416807       13.416816
61  5374  11.089691     11.089687       11.089687

[62 rows x 4 columns]


In [9]:
# def extract_decreasing_part(signal):
#     # Extract the time and depth arrays
#     times = signal['time']
#     depths = signal['depth']
#     
#     peak_index = np.argmax(depths)
# 
#     decreasing_times = times[peak_index:]
#     decreasing_depths = depths[peak_index:]
#     
#     return {'time': decreasing_times, 'depth': decreasing_depths}
# 
# df['decrease'] = df['signal'].apply(extract_decreasing_part)
# 
# def add_gaussian_noise_to_signal(signal, mean=0, std=1):
#     noisy_signal = signal.copy()
#     noisy_signal['depth'] = noisy_signal['depth'] + np.random.normal(mean, std, len(noisy_signal['depth']))
#     return noisy_signal
# 
# df['noisy_signal'] = df['decrease'].apply(lambda x: add_gaussian_noise_to_signal(x, mean=0, std=0.1))  # Adjust `std` as needed